In [ ]:
SEED = 42
N_RULETAKER = 800
N_COUNTERBENCH = 1000

import csv, json, random, re, time
from collections import defaultdict, Counter
from itertools import product

from datasets import load_dataset
from huggingface_hub import hf_hub_download

/home/z/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Internal representation


In [ ]:
import re
from collections import defaultdict
from itertools import product

from collections import defaultdict
from itertools import product


def is_var(x):
    return isinstance(x, str) and x.startswith('?')

def flip(lit):
    """negate a literal"""
    return (lit[0], lit[1], not lit[2])

def lit_to_str(lit):
    neg = '-' if lit[2] else ''
    return f"{neg}{lit[0]}({', '.join(lit[1])})"


def forward_chain(facts, rules, closed_world=True, max_rounds=40):

    pos = {f for f in facts if not f[2]}
    neg = {flip(f) for f in facts if f[2]}
    proof = {}

    domain = set()
    for f in facts:
        domain.update(f[1])
    for ants, cons in rules:
        for lit in ants + cons:
            domain.update(a for a in lit[1] if not is_var(a))
    domain = sorted(domain)

    rule_list = list(rules)
    var_cache = [sorted({a for lit in ants + cons for a in lit[1] if is_var(a)})
                 for ants, cons in rule_list]

    for _ in range(max_rounds):
        changed = False
        for (ants, cons), rvars in zip(rule_list, var_cache):

            for combo in product(domain, repeat=len(rvars)):
                sub = dict(zip(rvars, combo))
                ground_ants = []
                ok = True
                for p, args, is_neg in ants:
                    g = (p, tuple(sub[a] if is_var(a) else a for a in args), False)
                    if is_neg:
                        if closed_world:
                            ok = g not in pos
                        else:
                            ok = g in neg
                    else:
                        ok = g in pos and g not in neg
                    ground_ants.append((g, is_neg))
                    if not ok:
                        break
                if not ok:
                    continue
                for p, args, is_neg in cons:
                    g = (p, tuple(sub[a] if is_var(a) else a for a in args), False)
                    if is_neg:
                        if g not in neg:
                            neg.add(g)
                            changed = True
                    elif g not in pos and g not in neg:
                        pos.add(g)
                        proof[g] = (ants, cons, ground_ants)
                        changed = True
        if not changed:
            break
    return pos, neg, proof

def query_answer(facts, rules, query, closed_world=True):

    pos, neg, proof = forward_chain(facts, rules, closed_world)
    p, args, is_neg = query
    g = (p, args, False)
    if closed_world:
        holds = (g in pos) != is_neg
    else:
        if is_neg:
            holds = g in neg and g not in pos
        else:
            holds = g in pos and g not in neg
    return ('True' if holds else 'False'), pos, neg, proof


def proof_trace(lit, pos, neg, proof, depth=0, seen=None):

    if seen is None:
        seen = set()
    lines = ['  ' * depth + lit_to_str(lit)]
    if lit in proof and lit not in seen:
        seen.add(lit)
        ants, cons, ground_ants = proof[lit]
        rule_txt = (' & '.join(lit_to_str(a) for a in ants) + ' -> ' +
                    ' & '.join(lit_to_str(c) for c in cons))
        lines[-1] += f'   <- rule: {rule_txt}'
        for g, was_neg in ground_ants:
            if was_neg:
                lines.extend(['  ' * (depth + 1) + lit_to_str(g) + '   <- not derivable (negation as failure)'])
            else:
                lines.extend(proof_trace(g, pos, neg, proof, depth + 1, seen))
    elif lit in seen:
        lines[-1] += '   (already shown)'
    else:
        lines[-1] += '   <- given as a fact'
    return lines


Parsing natural language into facts and rules


In [ ]:
STOP_WORDS = {'the', 'a', 'an'}

def norm_const(text):
    words = [w for w in re.findall(r"[a-z0-9']+", text.lower()) if w not in STOP_WORDS]
    return '_'.join(words) if words else text

def norm_pred(text):
    text = text.lower().strip()
    text = re.sub(r"^(that|who|which)\s+", '', text)
    words = re.findall(r"[a-z0-9']+", text)
    out = []
    for w in words:
        if len(w) > 3 and w.endswith('s') and not w.endswith('ss'):
            w = w[:-1]
        out.append(w)
    return '_'.join(out)

ARTICLE = r'(?:a|an|the)\s+'

def split_and(text):
    parts = re.split(r'\s*,\s+|\s+and\s+', text)
    return [p.strip(' ,') for p in parts if p.strip(' ,')]

def parse_subj(subj, var):
    s = subj.strip().lower()
    if re.match(r'^(someone|somebody|something|anyone|anything|everything)$', s):
        return (var,)
    if re.match(r'^(they|them|it|he|she)$', s):
        return (var,)
    return (norm_const(s),)

def obj_arg(o, var):
    o = o.strip().lower()
    if re.match(r'^(someone|somebody|something|anyone|anything)\s+else$', o):
        return '?y'
    if re.match(r'^(someone|somebody|something|anyone|anything)$', o):
        return var
    if re.match(r'^(they|them|it|he|she|him|her)$', o):
        return var
    return norm_const(o)

def parse_binary(text, var, neg):

    words = text.split()
    if len(words) < 3:
        return None
    arts = [i for i, w in enumerate(words) if w in ('the', 'a', 'an')]
    if arts:
        i = arts[-1]
        if i < 2:
            return None
        subj, verb, obj = words[:i - 1], words[i - 1], words[i:]
        if len(subj) == 0:
            return None
        s_args = parse_subj(' '.join(subj), var)
        if s_args is None:
            return None
        return (norm_pred(verb), s_args + (obj_arg(' '.join(obj), var),), neg)
    toks = text.split()
    lead = 0
    while lead < len(toks) and toks[lead][:1].isupper():
        lead += 1
    trail = len(toks)
    while trail > lead and toks[trail - 1][:1].isupper():
        trail -= 1
    if 0 < lead < trail < len(toks):
        return (norm_pred(' '.join(toks[lead:trail])),
                parse_subj(' '.join(toks[:lead]), var) + (norm_const(' '.join(toks[trail:])),),
                neg)
    if len(words) == 3:
        s, verb, o = words
        s_args = parse_subj(s, var)
        if s_args is None:
            return None
        return (norm_pred(verb), s_args + (obj_arg(o, var),), neg)
    if len(words) == 4 and words[2] in ('someone', 'something'):
        s_args = parse_subj(' '.join(words[:2]), var)
        if s_args is None:
            return None
        return (norm_pred(words[1]), s_args + (obj_arg(' '.join(words[2:]), var),), neg)
    return None

def parse_literal(text, var):

    t = text.strip().rstrip('.').strip()
    t = re.sub(r'\s+', ' ', t)
    if not t:
        return None
    neg = False
    m = re.match(r'^(.*?)\s+is not\s+(.*)$', t) or re.match(r'^(.*?)\s+are not\s+(.*)$', t)
    if m:
        neg = True
    else:
        m = re.match(r'^(.*?)\s+is\s+(.*)$', t) or re.match(r'^(.*?)\s+are\s+(.*)$', t)
    if m:
        subj, rest = m.group(1).strip(), m.group(2).strip()
        if rest.startswith('not '):
            rest, neg = rest[4:].strip(), True
        rest = re.sub(rf'^{ARTICLE}', '', rest)
        pred = norm_pred(rest)
        if not pred:
            return None
        args = parse_subj(subj, var)
        if args is None:
            return None
        return (pred, args, neg)
    m = re.match(r'^(.*?)\s+(?:do|does)\s+not\s+(.*)$', t)
    if m:
        rest = m.group(1).strip() + ' ' + m.group(2).strip()
        return parse_binary(re.sub(r'\s+', ' ', rest), var, True)
    return parse_binary(t, var, neg)

def parse_parts(text, var):
    out = []
    prev_subj = None
    for frag in split_and(text):
        lit = parse_literal(frag, var)
        if lit is None and prev_subj and re.fullmatch(r'(?:not\s+)?[a-z]+', frag.strip()):
            # shared copula: '... is round and not red' -> '... is not red'
            lit = parse_literal(f'{prev_subj} is {frag.strip()}', var)
        if lit is not None:
            out.append(lit)
            sm = re.match(r'^(.*?)\s+(?:is|are|do|does)\b', frag.strip())
            if sm and sm.group(1).strip():
                prev_subj = sm.group(1).strip()
    return out

def parse_rule_sentence(sent):

    s = sent.strip().rstrip('.').strip()
    rules = []
    var = '?x'

    m = re.match(r'^if\s+(.*?),?\s+then\s+(.*)$', s, re.I)
    if m:
        ants = parse_parts(m.group(1), var)
        cons = parse_parts(m.group(2), var)
        if ants and cons:
            for c in cons:
                rules.append((tuple(ants), (c,)))
        return rules

    m = re.match(r'^(?!the\b)(?:all\s+|every\s+)?(.+?)\s+'
                 r'((?:[a-z]+s)|people|men|women|children|folk|cattle|sheep|deer|fish)\s+'
                 r'(?:are|is)\s+(.+)$', s, re.I)
    if m:
        left = m.group(1)
        conseq = m.group(3).strip()
        cons_neg = conseq.lower().startswith('not ')
        if cons_neg:
            conseq = conseq[4:]
        preds_l = [norm_pred(p) for p in split_and(left)]
        preds_c = [norm_pred(p) for p in split_and(conseq)]
        preds_l = [p for p in preds_l if p and p not in ('all', 'every')]
        ants = [(p, (var,), False) for p in preds_l]
        cons = [(p, (var,), cons_neg) for p in preds_c]
        if ants and cons:
            for c in cons:
                rules.append((tuple(ants), (c,)))
            return rules

    m = re.match(r'^(?:everyone|everybody|whoever|anyone)\s+who\s+(.*?)\s+(?:are|is)\s+(.+)$', s, re.I)
    if m:
        ants = parse_parts(m.group(1), var)
        conseq = m.group(2).strip()
        cons_neg = conseq.lower().startswith('not ')
        if cons_neg:
            conseq = conseq[4:]
        cons = [(norm_pred(conseq), (var,), cons_neg)]
        if ants:
            for c in cons:
                rules.append((tuple(ants), (c,)))
            return rules
    return rules

def parse_fact_sentence(sent):
    s = sent.strip().rstrip('.').strip()
    var = '?q'
    lit = parse_literal(s, var)
    if lit is None:
        return []
    if any(is_var(a) for a in lit[1]):
        return []
    return [lit]

def parse_compound_fact(sent):
    s = sent.strip().rstrip('.').strip()
    parts = re.split(r',\s+and\s+|;\s+', s)
    if len(parts) < 2:
        return None
    first = parse_fact_sentence(parts[0])
    if not first:
        return None
    out = list(first)
    subj = first[0][1]
    for p in parts[1:]:
        p = p.strip()
        p2 = re.sub(r'^(it|they|he|she)\s+', subj[0].replace('_', ' ') + ' ', p)
        lit = parse_fact_sentence(p2)
        if lit:
            out.extend(lit)
        else:
            return None
    return out

def parse_kb(text):
    facts, rules = set(), []
    for sent in re.split(r'(?<=\.)\s+', text.strip()):
        if not sent.strip():
            continue
        if re.match(r'^\s*(if|everyone|everybody|whoever|anyone)\b', sent, re.I):
            r = parse_rule_sentence(sent)
            rules.extend(r)
            continue
        r = parse_rule_sentence(sent)
        if r:
            rules.extend(r)
            continue
        f = parse_fact_sentence(sent)
        if not f:
            f2 = parse_compound_fact(sent)
            f = f2 if f2 is not None else []
        facts.update(f)
    return facts, rules

def parse_query(text):
    var = '?q'
    lit = parse_literal(text.strip(), var)
    if lit is None or any(is_var(a) for a in lit[1]):
        return None
    return lit


In [ ]:
kb = ("Bob is young. Bob is kind. "
      "If someone is young then they are smart. "
      "If someone is smart and kind then they are successful.")
facts, rules = parse_kb(kb)
q = parse_query("Bob is successful.")
answer, pos, neg, proof = query_answer(facts, rules, q, closed_world=True)
print("knowledge base:", kb)
print("query          : Bob is successful.")
print("answer         :", answer)
print()
print("reasoning trace:")
print("\n".join(proof_trace(q, pos, neg, proof)))

knowledge base: Bob is young. Bob is kind. If someone is young then they are smart. If someone is smart and kind then they are successful.
query          : Bob is successful.
answer         : True

reasoning trace:
successful(bob)   <- rule: smart(?x) & kind(?x) -> successful(?x)
  smart(bob)   <- rule: young(?x) -> smart(?x)
    young(bob)   <- given as a fact
  kind(bob)   <- given as a fact


FOLIO premises (open-world templates)

In [ ]:
HEAD_PLURALS = r'(?:[a-z]+s)|people|men|women|children|folk|cattle|sheep|deer|fish|offspring'

def skolem_counter():
    k = [0]
    def nxt():
        k[0] += 1
        return f'sk{k[0]}'
    return nxt

def parse_folio_sentence(sent, new_sk):
    s = sent.strip().rstrip('.').strip()
    if not s:
        return None
    var = '?x'

    if re.search(r'\b(either|neither|nor)\b', s, re.I) or re.search(r'\bor\b', re.sub(r'\b(honor|organ)\w*\b', '', s, flags=re.I)):
        return None

    m = re.match(r'^if\s+(.+?),?\s+then\s+(.+)$', s, re.I)
    if m:
        ants = parse_folio_conditions(m.group(1), var, new_sk)
        cons = parse_folio_conditions(m.group(2), var, new_sk)
        if ants and cons:
            return ('rule', [(tuple(ants), tuple(cons))])

    m = re.match(rf'^(all|every|any|no)\s+(.+?)\s+(?:are|is)\s+(not\s+)?(.+)$', s, re.I)
    if m:
        quant, np1, cons_neg, np2 = m.groups()
        head, rels = split_np(np1)
        ants = [(norm_pred(head), (var,), False)]
        ants += rel_preds(rels, var)
        cons_neg = bool(cons_neg) or quant.lower() == 'no'
        cons = [(norm_pred(np2), (var,), cons_neg)]
        return ('rule', [(tuple(ants), tuple(cons))])

    m = re.match(r'^some\s+(.+?)\s+(?:are|is)\s+(not\s+)?(.+)$', s, re.I)
    if m:
        np1, cons_neg, np2 = m.groups()
        k = new_sk()
        head, rels = split_np(np1)
        lits = [(norm_pred(head), (k,), False)] + rel_preds(rels, k)
        lits.append((norm_pred(np2), (k,), bool(cons_neg)))
        return ('fact', lits)
    m = re.match(r'^some\s+(.+?)\s+((?:do(?:es)?\s+not|cannot|can\s+not)\s+)?(.+)$', s, re.I)
    if m:
        np1, vp_neg, vp = m.groups()
        k = new_sk()
        head, rels = split_np(np1)
        lits = [(norm_pred(head), (k,), False)] + rel_preds(rels, k)
        lits.append((norm_pred(vp), (k,), bool(vp_neg)))
        return ('fact', lits)

    m = re.match(r'^([A-Z][\w\s]*?)\s+(?:and|or)\s+([A-Z][\w\s]*?)\s+are\s+(?:both\s+)?(not\s+)?(.+)$', s)
    if m and ' is ' not in s and ' are not ' not in s.lower():
        subj1, subj2, neg, np = m.groups()
        lits = []
        for subj in (subj1, subj2):
            for part in split_and(np):
                pn = part.strip()
                pneg = pn.lower().startswith('not ')
                if pneg:
                    pn = pn[4:]
                lits.append((norm_pred(strip_article(pn)), (norm_const(subj),), bool(neg) or pneg))
        return ('fact', lits)

    m = re.match(r'^(?:everyone|anyone|everybody|whoever|people)\s+who\s+(.+?)\s+((?:do(?:es)?\s+not|cannot)\s+)?(.+)$', s, re.I)
    if m and ' are ' not in s.lower() and ' is ' not in s.lower():
        cond, vp_neg, vp = m.groups()
        ants = parse_folio_conditions(cond, var, new_sk)
        if ants:
            cons = [(norm_pred(vp), (var,), bool(vp_neg))]
            return ('rule', [(tuple(ants), tuple(cons))])

    m = re.match(r'^([A-Z][\w\'\-]*(?:\s+[A-Z][\w\'\-]*)*)\s+(?:is|are)\s+(not\s+)?(.+)$', s)
    if m:
        subj, neg, rest = m.groups()
        lits = []
        ok = True
        for part in split_and(rest):
            pn = part.strip()
            pneg = pn.lower().startswith('not ')
            if pneg:
                pn = pn[4:]
            pn = strip_article(pn)
            if not pn:
                ok = False
                break
            lits.append((norm_pred(pn), (norm_const(subj),), bool(neg) or pneg))
        if ok and lits:
            return ('fact', lits)

    m = re.match(r'^([A-Z][\w\'\-]*(?:\s+[A-Z][\w\'\-]*)*)\s+((?:do(?:es)?\s+not|cannot|can\s+not|did\s+not|does\s+not)\s+)?(.+)$', s)
    if m:
        subj, vp_neg, vp = m.groups()
        if vp and vp.split()[0].lower() not in ('is', 'are', 'was', 'were'):
            return ('fact', [(norm_pred(vp), (norm_const(subj),), bool(vp_neg))])
    return None

def strip_article(np):
    return re.sub(r'^(a|an|the)\s+', '', np.strip(), flags=re.I)

def split_np(np1):
    """split 'people who regularly drink coffee' -> ('people', ['who regularly drink coffee'])"""
    m = re.search(r'\s+(who|that|which)\s+', np1, re.I)
    if not m:
        return np1.strip(), []
    return np1[:m.start()].strip(), [np1[m.start():].strip()]

def rel_preds(rels, var):
    out = []
    for r in rels:
        p = norm_pred(r)
        if p:
            out.append((p, (var,), False))
    return out

def parse_folio_conditions(text, var, new_sk):

    lits = parse_parts(text, var)
    if lits:
        return lits
    lit = parse_literal(text, var)
    if lit is not None and not any(is_var(a) is False and a.startswith('sk') for a in lit[1]):
        return [lit]
    p = norm_pred(text)
    return [(p, (var,), False)] if p else []


def folio_kb(premises):

    facts, rules = set(), []
    new_sk = skolem_counter()
    for sent in premises.split("\n"):
        sent = sent.strip()
        if not sent:
            continue
        parsed = parse_folio_sentence(sent, new_sk)
        if parsed is None:
            continue
        kind, payload = parsed
        if kind == "fact":
            facts.update(payload)
        else:
            rules.extend(payload)
    return facts, rules


 CounterBench: causal counterfactuals as rules

In [ ]:
def cb_split(info):
    info = re.sub(r'^We know that\s+', '', info.strip())
    obs = []
    m = re.search(r'\.\s+(We (?:did not )?observed?.*)$', info)
    if m:
        mm = re.match(r'We observed\s+(.*)$', m.group(1).strip().rstrip('.'))
        if mm:
            obs = [p.strip() for p in re.split(r'\s+and\s+', mm.group(1))]
        info = info[:m.start()]
    info = info.rstrip('.').strip()
    raw = re.split(r',\s+', info)
    clauses = []
    for seg in raw:
        seg = seg.strip()
        if seg.lower().startswith('and '):
            seg = seg[4:]
        if 'cause' not in seg and clauses:
            clauses[-1] += ', ' + seg
        else:
            clauses.append(seg)
    return clauses, obs

def cb_lit(s):
    s = s.strip().lower()
    if s.startswith('not '):
        return s[4:].strip(), True
    return s, False

def cb_clause_rules(clause):
    m = re.match(r'^(.+?)\s+causes?\s+(.+)$', clause.strip())
    if not m:
        return []
    lhs_txt = m.group(1).strip()
    had_together = lhs_txt.endswith('together')
    lhs_txt = re.sub(r'\s+together$', '', lhs_txt).strip()
    if ' and ' in lhs_txt:
        lhs = [cb_lit(x) for x in re.split(r'\s+and\s+', lhs_txt)]
        joiner = 'and'
    elif ' or ' in lhs_txt:
        lhs = [cb_lit(x) for x in re.split(r'\s+or\s+', lhs_txt)]
        joiner = 'or'
    else:
        lhs = [cb_lit(lhs_txt)]
        joiner = 'single'
    rhs = []
    for part in re.split(r'\s+and\s+', m.group(2).replace(', ', ' and ')):
        rhs.append(cb_lit(part))
    if joiner == 'or' and had_together:
        joiner = 'or'
    L = [('occurs', (v,), neg) for v, neg in lhs]
    R = [('occurs', (v,), neg) for v, neg in rhs]
    rules = []
    for r in R:
        if joiner == 'single':
            rules.append(((L[0],), (r,)))
            rules.append(((flip(L[0]),), (flip(r),)))
        elif joiner == 'and':
            rules.append((tuple(L), (r,)))
            for i in range(len(L)):
                rules.append(((flip(L[i]),), (flip(r),)))
        else:  # or
            for l in L:
                rules.append(((l,), (r,)))
            rules.append((tuple(flip(l) for l in L), (flip(r),)))
    return rules

def cb_parse_question(q, cb_type):
    q = q.strip().rstrip('?')
    if cb_type == 'nested':
        m = re.match(r'^Assume not (.+?), and based on this assumption, further suppose not (.+?)\. Would (.+?) occur$', q)
        if not m:
            return None
        return [('not', m.group(1)), ('not', m.group(2))], m.group(3)
    m = re.match(r'^Would (.+?) occur if (.+?)$', q)
    if not m:
        return None
    conds = m.group(2).replace(' instead of ', ' IFSEP ')
    ints = []
    for part in re.split(r'\s+and\s+', conds):
        pm = re.match(r'^not (.+?)(?: IFSEP (.+))?$', part.strip())
        ints.append(('not', pm.group(1)) if pm else ('pos', part.strip()))
    return ints, m.group(1)

def cb_predict(ex):
    """returns 'yes'/'no'"""
    clauses, obs = cb_split(ex['given_info'])
    pq = cb_parse_question(ex['question'], ex['type'])
    if pq is None:
        return 'no'
    ints, target = pq
    int_vars = {v.lower() for _, v in ints}
    rules = []
    for c in clauses:
        for ants, cons in cb_clause_rules(c):
            if any(r[1][0] in int_vars for r in cons):
                continue
            rules.append((ants, cons))
    facts = {('occurs', (v.lower(),), pol == 'not') for pol, v in ints}
    for o in obs:
        v, neg = cb_lit(o)
        facts.add(('occurs', (v,), neg))
        if v not in int_vars:
            int_vars.add(v)
            rules = [r for r in rules if not any(c[1][0] == v for c in r[1])]
    q = ('occurs', (target.lower(),), False)
    ans, pos, neg, proof = query_answer(facts, rules, q, closed_world=False)
    return 'yes' if ans == 'True' else 'no'


Predict functions and evaluation sample

In [ ]:
def predict_ruletaker(ex):
    facts, rules = parse_kb(ex["context"])
    q = parse_query(ex["question"])
    if q is None:
        return "False"
    answer, pos, neg, proof = query_answer(facts, rules, q, closed_world=True)
    return answer

def predict_folio(ex):
    facts, rules = folio_kb(ex["premises"])
    q = parse_query(ex["conclusion"])
    if q is None:
        return "False"
    answer, pos, neg, proof = query_answer(facts, rules, q, closed_world=False)
    return answer

def predict_counterbench(ex):
    return "True" if cb_predict(ex) == "yes" else "False"

In [ ]:
# load the three public dataset

rt = load_dataset("tasksource/ruletaker")
fo = load_dataset("tasksource/folio")
cb_file = hf_hub_download(repo_id="CounterBench/CounterBench",
                          filename="data_balanced_alpha_V1.json",
                          repo_type="dataset")
with open(cb_file) as f:
    cb = json.load(f)

random.seed(SEED)
rt_sample = [{"context": rt["test"][i]["context"],
              "question": rt["test"][i]["question"],
              "gold": "True" if rt["test"][i]["label"] == "entailment" else "False"}
             for i in random.sample(range(len(rt["test"])), N_RULETAKER)]

random.seed(SEED)
cb_idx = random.sample(range(len(cb)), N_COUNTERBENCH)
cb_sample = [{"given_info": cb[i]["given_info"],
              "question": cb[i]["question"],
              "type": cb[i]["type"],
              "gold": "True" if cb[i]["answer"] == "yes" else "False"}
             for i in sorted(cb_idx)]

fo_sample = [{"premises": ex["premises"],
              "conclusion": ex["conclusion"],
              "gold": "True" if ex["label"] == "True" else "False"}
             for ex in fo["validation"]]

print("samples..evaluation")
print("  ruletaker    :", len(rt_sample), "queries (test split)")
print("  counterbench :", len(cb_sample), "queries (from 1,000)")
print("  folio        :", len(fo_sample), "queries (full validatinn set)")

evaluation samples:
  ruletaker    : 800 queries (sampled from the test split)
  counterbench : 1000 queries (sampled from the 1,000)
  folio        : 203 queries (full validation set)


## 6. Run the evaluation — accuracy and inference time

In [ ]:
results = {}
rows = []
t_start = time.time()
for name, examples, predict_fn in [
    ("ruletaker",     rt_sample, predict_ruletaker),
    ("counterbench",  cb_sample, predict_counterbench),
    ("folio",         fo_sample, predict_folio),
]:
    t0 = time.time()
    correct = 0
    for i, ex in enumerate(examples):
        pred = predict_fn(ex)
        if pred == ex["gold"]:
            correct += 1
        rows.append({"id": f"{name}_{i:04d}", "prediction": pred})
    dt = time.time() - t0
    results[name] = (len(examples), correct, dt)

t_total = time.time() - t_start

print(f"{'dataset':<14}{'queries':>8}{'correct':>9}{'accuracy':>10}{'time':>9}")
print("-" * 50)
n_all, c_all = 0, 0
for name, (n, c, dt) in results.items():
    print(f"{name:<14}{n:>8}{c:>9}{100*c/n:>9.1f}%{dt:>8.2f}s")
    n_all += n
    c_all += c
print("-" * 50)
print(f"{'TOTAL':<14}{n_all:>8}{c_all:>9}{100*c_all/n_all:>9.1f}%{t_total:>8.2f}s")
print(f"\nfinal accuracy: {100*c_all/n_all:.1f}%   "
      f"total inference time: {t_total:.2f}s "
      f"({1000*t_total/n_all:.1f} ms per query)")

dataset        queries  correct  accuracy     time
--------------------------------------------------
ruletaker          800      800    100.0%    0.35s
counterbench      1000      960     96.0%    0.11s
folio              203      132     65.0%    0.03s
--------------------------------------------------
TOTAL             2003     1892     94.5%    0.49s

final accuracy: 94.5%   total inference time: 0.49s (0.2 ms per query)


Reasoning traces


In [ ]:
import re as _re

ex = next(e for e in rt_sample
          if e["gold"] == "True" and "not" not in e["question"]
          and len(parse_kb(e["context"])[1]) >= 5)
facts, rules = parse_kb(ex["context"])
q = parse_query(ex["question"])
answer, pos, neg, proof = query_answer(facts, rules, q, closed_world=True)
print("RULETAKER STORY")
for s in _re.split(r"(?<=\.)\s+", ex["context"]):
    print("   ", s.strip())
print("QUERY:", ex["question"], " gold:", ex["gold"], " predicted:", answer)
print("trace:")
print("\n".join(proof_trace(q, pos, neg, proof)))

print()
print("=" * 70)

ex2 = next(e for e in rt_sample
           if " not " in e["question"] and e["gold"] == "True"
           and parse_query(e["question"]) is not None
           and parse_query(e["question"])[2]
           and query_answer(*parse_kb(e["context"]), parse_query(e["question"]), closed_world=True)[0] == "True")
facts2, rules2 = parse_kb(ex2["context"])
q2 = parse_query(ex2["question"])
answer2, pos2, neg2, proof2 = query_answer(facts2, rules2, q2, closed_world=True)
print("RULETAKER STORY (negated query)")
for s in _re.split(r"(?<=\.)\s+", ex2["context"]):
    print("   ", s.strip())
print("QUERY:", ex2["question"], " gold:", ex2["gold"], " predicted:", answer2)
print("trace:")
print(lit_to_str(q2), " holds because", lit_to_str(flip(q2)),
      "is NOT derivable (negation as failure):")
print("   known facts with that predicate:",
      sorted({lit_to_str(l) for l in pos2 if l[0] == flip(q2)[0]}) or "none")

print()
print("=" * 70)

ex3 = next(e for e in cb_sample if e["gold"] == "False")
clauses, obs = cb_split(ex3["given_info"])
print("COUNTERBENCH QUESTION")
print("    mechanisms  :", ex3["given_info"])
print("    question    :", ex3["question"], " gold:", ex3["gold"])
ints, target = cb_parse_question(ex3["question"], ex3["type"])
print("    intervention:", "; ".join(f"{v} = {'false' if p == 'not' else 'true'}" for p, v in ints))
rules3 = []
int_vars = {v.lower() for _, v in ints}
for c in clauses:
    for ants, cons in cb_clause_rules(c):
        if any(r[1][0] in int_vars for r in cons):
            continue
        rules3.append((ants, cons))
facts3 = {("occurs", (v.lower(),), pol == "not") for pol, v in ints}
ans3, pos3, neg3, proof3 = query_answer(facts3, rules3, ("occurs", (target.lower(),), False), closed_world=False)
print("    causal rules after truncating the intervened variable:")
for ants, cons in rules3:
    print("       ", " & ".join(lit_to_str(a) for a in ants), "->", lit_to_str(cons[0]))
print("    derived:", target.lower(), "= occurs" if ans3 == "True" else "= does NOT occur")
print("    answer  :", ans3)

RULETAKER STORY
    The bald eagle is round.
    The bald eagle needs the lion.
    The bald eagle needs the squirrel.
    The bald eagle visits the lion.
    The cat eats the bald eagle.
    The cat is blue.
    The cat is nice.
    The lion eats the cat.
    The lion is nice.
    The squirrel does not eat the cat.
    The squirrel is nice.
    The squirrel visits the bald eagle.
    If someone is round then they do not need the cat.
    If someone is nice and they need the lion then they visit the squirrel.
    If someone eats the squirrel then they are blue.
    If someone visits the lion then they eat the lion.
    If someone visits the squirrel then the squirrel is young.
    If someone is young and they need the lion then the lion visits the cat.
    If someone visits the cat then they need the squirrel.
    If the lion is blue then the lion eats the squirrel.
    If someone is nice then they need the lion.
QUERY: The lion visits the cat.  gold: True  predicted: True
trace:
visit

CSV Submission file export

In [ ]:
with open("submission.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["id", "prediction"])
    writer.writeheader()
    writer.writerows(rows)

print(f"wrote submission.csv with {len(rows)} predictions")
import pandas as pd
print(pd.read_csv("submission.csv").head(8).to_string(index=False))

wrote submission.csv with 2003 predictions
            id  prediction
ruletaker_0000       False
ruletaker_0001       False
ruletaker_0002       False
ruletaker_0003       False
ruletaker_0004        True
ruletaker_0005       False
ruletaker_0006        True
ruletaker_0007        True
